# Retrieval-Augmented Generation (RAG) for PDF Documents

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 14 — Question Answering and Information Retrieval  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Extract and chunk text from PDF files using PyMuPDF
2. Build a FAISS vector index from TF-IDF document embeddings
3. Retrieve the top-k most relevant chunks for a user query
4. Generate an answer by augmenting the LLM prompt with retrieved context


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install openai pymupdf faiss-cpu scikit-learn
```


In [ ]:
# pip install openai pymupdf faiss-cpu scikit-learn  # run once in terminal
import os
from pathlib import Path

import fitz  # PyMuPDF
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

import openai

DIR_PDF = Path.cwd() / "files"


---

## 2. PDF Text Extraction

PyMuPDF (`fitz`) reads each page of a PDF and concatenates the text.


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract and concatenate text from all pages of a PDF file."""
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text() for page in doc)


---

## 3. Text Chunking

Long documents are split into overlapping chunks so that individual retrieval  
units fit within the LLM context window.


In [ ]:
def extract_and_chunk_text(
    pdf_path: Path,
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> list[str]:
    """Extract text from a PDF and split it into overlapping character chunks."""
    full_text = extract_text_from_pdf(pdf_path)
    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunks.append(full_text[start:end])
        start += chunk_size - chunk_overlap
    return chunks


---

## 4. FAISS Vector Index

TF-IDF converts each chunk into a sparse vector; FAISS indexes these vectors  
for approximate nearest-neighbour retrieval.


In [ ]:
def create_index(
    documents: list[str],
) -> tuple[faiss.IndexFlatL2, TfidfVectorizer]:
    """Fit a TF-IDF vectoriser and build a FAISS L2 index over the document chunks."""
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(documents).toarray().astype("float32")
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    return index, vectorizer


---

## 5. Retrieve Relevant Chunks


In [ ]:
def search_documents(
    query: str,
    index: faiss.IndexFlatL2,
    vectorizer: TfidfVectorizer,
    documents: list[str],
    top_k: int = 3,
) -> list[str]:
    """Return the top-k document chunks most similar to *query*."""
    query_vec = vectorizer.transform([query]).toarray().astype("float32")
    _, indices = index.search(query_vec, top_k)
    return [documents[i] for i in indices[0] if i < len(documents)]


---

## 6. Generate an Augmented Response

Retrieved chunks are prepended to the user query as grounding context,  
reducing hallucinations and keeping the answer anchored to the source document.


In [ ]:
def generate_augmented_response(
    query: str,
    index: faiss.IndexFlatL2,
    vectorizer: TfidfVectorizer,
    documents: list[str],
    api_key: str,
) -> str:
    """Retrieve relevant chunks and generate an OpenAI-backed augmented answer."""
    openai.api_key = api_key
    relevant_chunks = search_documents(query, index, vectorizer, documents)
    context = "\n\n".join(relevant_chunks)

    augmented_prompt = (
        "Answer the question based on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": augmented_prompt}],
        max_tokens=500,
    )
    return response.choices[0].message.content


---

## 7. Main Workflow


In [ ]:
def main() -> None:
    """Run the full RAG pipeline over all PDFs in DIR_PDF."""
    api_key = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")

    # Load and chunk all PDF files in the data folder
    all_chunks: list[str] = []
    for pdf_file in DIR_PDF.glob("*.pdf"):
        print(f"Loading: {pdf_file.name}")
        all_chunks.extend(extract_and_chunk_text(pdf_file))

    print(f"Total chunks: {len(all_chunks)}")

    # Build the vector index
    index, vectorizer = create_index(all_chunks)

    # Interactive query loop
    while True:
        query = input("Enter your question (or 'quit'): ").strip()
        if query.lower() in ("quit", "q", ""):
            break
        answer = generate_augmented_response(query, index, vectorizer, all_chunks, api_key)
        print(f"\nAnswer: {answer}\n")


# Uncomment to run in a terminal; interactive input is not supported inside Jupyter
# main()


> **Output interpretation:** The pipeline loads all PDFs from DIR_PDF, splits them into 1000-character chunks (200-character overlap), indexes them with FAISS, and waits for queries. Each query retrieves the 3 most similar chunks by TF-IDF cosine distance, then sends them as context to GPT-3.5-turbo. For better retrieval quality, replace TF-IDF with a dense embedding model such as `sentence-transformers/all-MiniLM-L6-v2`.


---

## Summary

| Component | Implementation | Upgrade path |
|---|---|---|
| Text extraction | PyMuPDF (`fitz`) | Same; add OCR for scanned PDFs |
| Chunking | Character-based with overlap | Sentence or paragraph splitting |
| Embedding | TF-IDF | Dense embeddings (Sentence-BERT, OpenAI `text-embedding-3`) |
| Index | FAISS L2 | FAISS HNSW for scalability; Pinecone/Weaviate for persistence |
| Generation | OpenAI GPT-3.5-turbo | Any instruction-tuned LLM |
